In [24]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
import yfinance as yf
import ta
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning, ValueWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=ValueWarning)

# Load full dataset
df = pd.read_csv(r"..\Week1\week1.csv")
df.dropna(inplace=True)
df['Date'] = pd.to_datetime(df['Date'])

tickers = df['Ticker'].unique()

for ticker in tickers:
    print(f"\nProcessing {ticker}...")
    stock_df = df[df['Ticker'] == ticker].dropna().reset_index(drop=True)
    stock_df['Volume'] = np.log1p(stock_df['Volume'])
    stock_df['Volume_avg_7'] = np.log1p(stock_df['Volume_avg_7'])
    stock_df['Volume_lag_1'] = np.log1p(stock_df['Volume_lag_1'])

    stock_df['Date'] = pd.to_datetime(stock_df['Date'])
    stock_df = stock_df.sort_values('Date')
    stock_df.set_index('Date', inplace=True)

    def full_decompose(series, model='additive', period=30):
        decomposition = seasonal_decompose(series, model=model, period=period, extrapolate_trend='freq')
        return decomposition.trend, decomposition.seasonal, decomposition.resid

    close_trend, close_seasonal, close_resid = full_decompose(stock_df['Close'], model='additive', period=30)
    stock_df['Close_trend'] = close_trend
    stock_df['Close_seasonal'] = close_seasonal
    stock_df['Close_resid'] = close_resid

    volume_trend, volume_seasonal, volume_resid = full_decompose(stock_df['Volume'], model='additive', period=30)
    stock_df['Volume_trend'] = volume_trend
    stock_df['Volume_seasonal'] = volume_seasonal
    stock_df['Volume_resid'] = volume_resid
    stock_df.dropna(inplace=True)
    stock_df.reset_index(inplace=True)

    stock_df['Date'] = pd.to_datetime(stock_df['Date'])
    stock_df.set_index('Date', inplace=True)
    
    start_date = stock_df.index.min().strftime('%Y-%m-%d')
    end_date = stock_df.index.max().strftime('%Y-%m-%d')

    symbols = {
        'SP500': '^GSPC',
        'NASDAQ': '^IXIC',
        'VIX': '^VIX'
    }

    external_data = pd.DataFrame()
    for name, symbol in symbols.items():
        temp = yf.download(symbol, start=start_date, end=end_date, progress=False)[['Close']]
        
        if isinstance(temp.columns, pd.MultiIndex):
            temp.columns = temp.columns.get_level_values(0)

        temp = temp.rename(columns={'Close': f'{name}_Close'})
        temp[f'{name}_Return'] = temp[f'{name}_Close'].pct_change()

        external_data = pd.concat([external_data, temp[[f'{name}_Return']]], axis=1)

    external_data.index = pd.to_datetime(external_data.index)
    stock_df.index = pd.to_datetime(stock_df.index)
    stock_df = stock_df.join(external_data, how='left')

    stock_df.dropna(inplace=True)
    stock_df.reset_index(inplace=True)
    stock_df['ADX'] = ta.trend.adx(stock_df['High'], stock_df['Low'], stock_df['Close'], window=14)

    macd = ta.trend.macd(stock_df['Close'], window_slow=26, window_fast=12)
    macd_signal = ta.trend.macd_signal(stock_df['Close'], window_slow=26, window_fast=12, window_sign=9)
    stock_df['MACD'] = macd
    stock_df['MACD_Signal'] = macd_signal
    stock_df.dropna(inplace=True)

    features = [
    'Close_lag_1', 'Close_lag_2', 'Close_lag_3',
    'Daily Return', 'Return_lag_1', 'Return_lag_2',
    'MA_7', 'MA_30', 'STD_7', 'STD_30',
    'Volume_lag_1', 'Volume_avg_7', 'Volume_change',
    '30D RV', 'MA_ratio', 'HL_ratio', 'CO_ratio',
    'Close_trend', 'Close_seasonal', 'Close_resid',
    'Volume_trend', 'Volume_seasonal', 'Volume_resid',
    'SP500_Return', 'NASDAQ_Return', 'VIX_Return', 'ADX',
    'MACD', 'MACD_Signal'
    ]

    stock_df['Date'] = pd.to_datetime(stock_df['Date'])
    stock_df = stock_df.set_index('Date')

    split = int(0.8 * len(stock_df))
    train_df = stock_df.iloc[:split]
    test_df = stock_df.iloc[split:]
    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df['Target']
    y_test = test_df['Target']

    model = SARIMAX(
    endog=y_train,
    exog=X_train,
    enforce_stationarity=False,
    enforce_invertibility=False
    )

    results = model.fit(disp=False)
    y_pred = results.predict(
    start=len(X_train),
    end=len(X_train) + len(X_test) - 1,
    exog=X_test
    )
    y_pred.index = y_test.index

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    print(f"ARIMAX Model RMSE on Test Set: {rmse:.6f}")
    print(f"ARIMAX Model MAE on Test Set: {mae:.6f}")
    # Predict on training set
    y_train_pred = results.predict(
    start=X_train.index[0],
    end=X_train.index[-1],
    exog=X_train
    )

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    print(f"ARIMAX Model RMSE on Train Set: {train_rmse:.6f}")
    print(f"ARIMAX Model MAE on Train Set: {train_mae:.6f}")

    close_train_today = stock_df.loc[X_train.index, 'Close'].values
    close_test_today = stock_df.loc[X_test.index, 'Close'].values

    predicted_close_train_tomorrow = close_train_today * (1 + y_train_pred)
    predicted_close_test_tomorrow = close_test_today * (1 + y_pred)

    true_close_train_tomorrow = stock_df.loc[y_train.index, 'Close'].shift(-1).values
    true_close_test_tomorrow = stock_df.loc[y_test.index, 'Close'].shift(-1).values

    predicted_close_train_tomorrow = predicted_close_train_tomorrow[:-1]
    true_close_train_tomorrow = true_close_train_tomorrow[:-1]

    predicted_close_test_tomorrow = predicted_close_test_tomorrow[:-1]
    true_close_test_tomorrow = true_close_test_tomorrow[:-1]

    train_price_mae = mean_absolute_error(true_close_train_tomorrow, predicted_close_train_tomorrow)
    train_price_rmse = np.sqrt(mean_squared_error(true_close_train_tomorrow, predicted_close_train_tomorrow))

    test_price_mae = mean_absolute_error(true_close_test_tomorrow, predicted_close_test_tomorrow)
    test_price_rmse = np.sqrt(mean_squared_error(true_close_test_tomorrow, predicted_close_test_tomorrow))

    print(f"Close Price Prediction MAE (Train): {train_price_mae:.6f}")
    print(f"Close Price Prediction RMSE (Train): {train_price_rmse:.6f}")
    print(f"Close Price Prediction MAE (Test): {test_price_mae:.6f}")
    print(f"Close Price Prediction RMSE (Test): {test_price_rmse:.6f}")

    train_direction_acc = (np.sign(y_train_pred) == np.sign(y_train)).mean()
    test_direction_acc = (np.sign(y_pred) == np.sign(y_test)).mean()

    print("Direction Accuracy (Train):", train_direction_acc)
    print("Direction Accuracy (Test):", test_direction_acc)
    
    result_df = pd.DataFrame({
        'Date': y_test.index[:-1],  # exclude last due to shift
        'Actual Close': true_close_test_tomorrow,
        'Predicted Close': predicted_close_test_tomorrow
    })
    result_df.to_csv(f"{ticker}_predictions.csv", index=False)


Processing AAPL...
ARIMAX Model RMSE on Test Set: 0.013508
ARIMAX Model MAE on Test Set: 0.009023
ARIMAX Model RMSE on Train Set: 0.015045
ARIMAX Model MAE on Train Set: 0.011164
Close Price Prediction MAE (Train): 0.849461
Close Price Prediction RMSE (Train): 1.187197
Close Price Prediction MAE (Test): 1.191102
Close Price Prediction RMSE (Test): 1.896713
Direction Accuracy (Train): 0.6373779637377964
Direction Accuracy (Test): 0.6239554317548747

Processing AMZN...
ARIMAX Model RMSE on Test Set: 0.017864
ARIMAX Model MAE on Test Set: 0.011803
ARIMAX Model RMSE on Train Set: 0.019194
ARIMAX Model MAE on Train Set: 0.013591
Close Price Prediction MAE (Train): 4.328547
Close Price Prediction RMSE (Train): 6.675298
Close Price Prediction MAE (Test): 10.542043
Close Price Prediction RMSE (Test): 16.976124
Direction Accuracy (Train): 0.6192468619246861
Direction Accuracy (Test): 0.6239554317548747

Processing GOOGL...
ARIMAX Model RMSE on Test Set: 0.010176
ARIMAX Model MAE on Test Set: 0

In [7]:
# # Basic Signal: Buy if predicted close > today's close
# test_signals = (predicted_close_test_tomorrow > close_test_today[:-1]).astype(int)  # 1 = Buy, 0 = Sell
# # Entry: today's close, Exit: predicted tomorrow's close
# entry_prices = close_test_today[:-1]
# exit_prices = predicted_close_test_tomorrow

# # Real exit prices (ground truth, not predicted)
# true_exit_prices = true_close_test_tomorrow

# # Stop-loss logic: 5% loss from entry → exit at stop-loss price
# stop_loss_pct = 0.05
# stop_loss_price = entry_prices * (1 - stop_loss_pct)

# # Apply stop-loss: If true price falls below stop-loss, exit at stop-loss
# final_exit_prices = np.where(true_exit_prices < stop_loss_price, stop_loss_price, true_exit_prices)

# # Calculate returns only when signal == 1 (buy signal)
# returns = np.where(test_signals == 1, (final_exit_prices - entry_prices) / entry_prices, 0.0)
# # Assume risk-free rate ≈ 0, daily returns
# mean_return = np.mean(returns)
# std_return = np.std(returns)
# sharpe_ratio = (mean_return / std_return) * np.sqrt(252)  # Annualized
# print(f"Total Trades: {test_signals.sum()}")
# print(f"Average Return per Trade: {mean_return:.4f}")
# print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
# print(f"Total Return: {np.sum(returns) * 100:.2f}%")
# print(f"Max Drawdown: {(np.min(np.cumsum(returns)) * 100):.2f}%")